In [1]:
import pandas as pd
import folium
from folium.plugins import HeatMap

# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv("impact_dataset.csv")

print("Dataset Loaded")
print(df.shape)

# ==========================================
# Create Base Map
# ==========================================

center_lat = df["latitude"].mean()
center_lon = df["longitude"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    control_scale=True
)

# ==========================================
# Impact Weighted Heatmap
# ==========================================

heat_data = []

for _, row in df.iterrows():

    heat_data.append([
        row["latitude"],
        row["longitude"],
        row["impact_score"]
    ])

HeatMap(
    heat_data,
    radius=15,
    blur=10,
    max_zoom=13
).add_to(m)

print("Heatmap Added")

# ==========================================
# Junction Hotspots
# ==========================================

junction_hotspots = (
    df[df["junction"].notna()]
    .groupby("junction")
    .agg(
        event_count=("id", "count"),
        avg_impact=("impact_score", "mean"),
        latitude=("latitude", "mean"),
        longitude=("longitude", "mean")
    )
    .reset_index()
)

junction_hotspots = junction_hotspots.sort_values(
    by="event_count",
    ascending=False
)

top_hotspots = junction_hotspots.head(30)

# ==========================================
# Save Hotspots Table
# ==========================================

top_hotspots.to_csv(
    "top_hotspots.csv",
    index=False
)

# ==========================================
# Risk Classification
# ==========================================

def get_risk(impact):

    if impact >= 75:
        return "HIGH"

    elif impact >= 55:
        return "MEDIUM"

    else:
        return "LOW"


# ==========================================
# Resource Recommendation
# ==========================================

def recommend_actions(impact):

    if impact >= 75:

        return {
            "resource_category": "HIGH",
            "actions": [
                "Deploy Traffic Response Team",
                "Install Temporary Barricades",
                "Activate Diversion Route",
                "Continuous Monitoring"
            ]
        }

    elif impact >= 55:

        return {
            "resource_category": "MEDIUM",
            "actions": [
                "Deploy Additional Traffic Personnel",
                "Place Temporary Barricades",
                "Monitor Traffic Flow",
                "Prepare Alternate Route Advisory"
            ]
        }

    else:

        return {
            "resource_category": "LOW",
            "actions": [
                "Routine Monitoring",
                "Periodic Patrol Check",
                "No Diversion Required"
            ]
        }

# ==========================================
# Pin Color
# ==========================================

def get_pin_color(impact):

    if impact >= 80:
        return "red"

    elif impact >= 60:
        return "orange"

    else:
        return "green"


# ==========================================
# Add Top 10 Hotspot Pins
# ==========================================

for idx, row in top_hotspots.iterrows():

    recommendation = recommend_actions(
    row["avg_impact"]
    )

    risk = get_risk(
        row["avg_impact"]
    )

    actions_html = "<br>".join(
        [f"• {action}" for action in recommendation["actions"]]
    )

    popup_text = f"""
    <b>HOTSPOT</b><br><br>

    Junction: {row['junction']}<br>

    Events: {row['event_count']}<br>

    Traffic Impact Index:
    {row['avg_impact']:.2f}<br>

    Risk Level:
    {risk}<br>

    Resource Category:
    {recommendation['resource_category']}<br><br>

    <b>Recommended Actions</b><br>
    {actions_html}
    """

    folium.Marker(
        location=[
            row["latitude"],
            row["longitude"]
        ],
        popup=folium.Popup(
            popup_text,
            max_width=300
        ),
        tooltip=f"{row['junction']}",
        icon=folium.Icon(
            color=get_pin_color(
                row["avg_impact"]
            ),
            icon="info-sign"
        )
    ).add_to(m)

print("Top Hotspot Pins Added")

# ==========================================
# Save Map
# ==========================================

output_file = "traffic_hotspots_final.html"

m.save(output_file)

print("\nMap Saved Successfully")
print(output_file)

# ==========================================
# Display Top 10
# ==========================================

print("\nTop 10 Junction Hotspots\n")

print(
    top_hotspots[
        [
            "junction",
            "event_count",
            "avg_impact"
        ]
    ]
)

Dataset Loaded
(8173, 53)
Heatmap Added
Top Hotspot Pins Added

Map Saved Successfully
traffic_hotspots_final.html

Top 10 Junction Hotspots

                           junction  event_count  avg_impact
175                    MekhriCircle           64   65.375000
20                AyyappaTempleJunc           49   58.734694
241           SatteliteBusStandJunc           43   66.046512
292             YeshwanthpuraCircle           38   64.631579
290                  YelhankaCircle           34   62.441176
293           toll gate mysore road           33   65.515152
251                   SilkBoardJunc           33   65.242424
122       JalahalliCross(SM Circle)           32   64.562500
193           Nagavara-ORR Junction           32   64.531250
132                  KIMCO Junction           31   64.483871
128                      K R Circle           31   67.032258
279   VeerannapalyaJunction(BEL,HO)           30   64.400000
268                TownhallJunction           30   67.133333
72  